# Variable-Count Flare Inference with PCAT

This notebook uses Ardor's synthetic Tier 2 light-curve example and passes its CSV input through `tier3_pcat()`. PCAT samples a variable number of flares; the posterior count distribution and full-baseline model fit are saved under `visuals/`.

The light curve, noise, and fit are all synthetic. They exercise the inference pipeline only and say nothing about a real star or flare-recovery rate. The smoke configuration uses one process and 1,200 sweeps so that the example remains small enough to run locally. This short chain does not test convergence and need not visit the injected flare count.

In [ ]:
from datetime import datetime
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

warnings.filterwarnings(
    "ignore",
    message="Unable to find acceptable character detection dependency.*",
)

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
repository_root = next(
    candidate for candidate in search_roots
    if (candidate / "setup.py").is_file()
    and (candidate / "src" / "ardor").is_dir()
)
example_root = repository_root / "examples"
for import_root in (
    repository_root / "src",
    repository_root.parent.parent / "tansudaylan" / "tdpy",
    example_root,
):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from ardor.Flares.pcat_flares import tier3_pcat
from pcat_variable_flares import (
    REFERENCE_BJD,
    save_fit_figures,
    simulate_light_curve,
)
from tdpy.verbosity import print

visuals_directory = example_root / "visuals"
visuals_directory.mkdir(exist_ok=True)
run_name = datetime.now().strftime("notebook_pcat_flares_%Y%m%d_%H%M%S")
rng_seed = 11

In [ ]:
time_days, flux, uncertainty, injected_peak_times = simulate_light_curve(
    seed=rng_seed,
)
input_figure, input_axis = plt.subplots(figsize=(8.0, 3.6), constrained_layout=True)
time_hours = (time_days - time_days[0]) * 24.0  # [hour]
input_axis.errorbar(
    time_hours,
    flux,
    yerr=uncertainty,
    fmt=".",
    ms=2.5,
    lw=0.5,
    color="#454545",
    label="Synthetic observed flux",
)
for index, peak_time in enumerate(injected_peak_times):
    input_axis.axvline(
        (peak_time - time_days[0]) * 24.0,
        color="#A51417",
        linestyle=":" ,
        linewidth=0.9,
        label="Injected peak" if index == 0 else None,
    )
input_axis.set_xlabel("Time [hour]")
input_axis.set_ylabel("Relative flux")
input_axis.legend(loc="upper right")
input_axis.grid(False)
input_figure.patch.set_facecolor("white")
input_path = visuals_directory / f"ardor_pcat_variable_flare_input_{run_name}.png"
print(f"Writing to visuals/{input_path.name}...")
input_figure.savefig(input_path, dpi=250, bbox_inches="tight", facecolor="white")
display(input_figure)
plt.close(input_figure)
print(f"Injected flare count: {len(injected_peak_times)}.")

In [ ]:
input_directory = example_root / "data" / "inpt" / run_name
input_directory.mkdir(parents=True, exist_ok=True)
tier2_path = input_directory / "Flare_1.csv"
print(f"Writing to examples/data/inpt/{run_name}/Flare_1.csv...")
np.savetxt(
    tier2_path,
    np.column_stack((time_days, flux, uncertainty)),
    delimiter=",",
    header=str(REFERENCE_BJD),
)

output_directory = example_root / "data" / "outp" / run_name
fits = tier3_pcat(
    input_directory,
    output_directory,
    host_name="Synthetic_Star",
    numbswep=1200,
    max_flares=10,
    numbproc=1,
)
fit = fits[0]
figure_paths = save_fit_figures(fit, injected_peak_times, run_name)
sampled_flare_counts = np.asarray(
    fit.posterior.listpostnumbelem, dtype=int
).ravel()
print(
    f"PCAT sampled flare-count values: "
    f"{sorted(set(sampled_flare_counts))}."
)
for figure_path in figure_paths:
    display(Image(filename=str(figure_path)))